<a href="https://colab.research.google.com/github/gretadive/correlacion_actividad_solar_elnino/blob/main/05_limpieza_precipitacion_ERA5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install openpyxl

In [9]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [10]:
ruta = "7666a0490891a08eeb53407c7d1c8235.nc"

ds = xr.open_dataset(ruta)

print(ds)
print("\nInformación de la variable tp:")
print(ds["tp"].attrs)

<xarray.Dataset> Size: 8MB
Dimensions:     (valid_time: 920, latitude: 51, longitude: 41)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 7kB 1950-01-01 ... 2026-08-01
  * latitude    (latitude) float64 408B -3.0 -3.1 -3.2 -3.3 ... -7.8 -7.9 -8.0
  * longitude   (longitude) float64 328B -82.0 -81.9 -81.8 ... -78.2 -78.1 -78.0
    number      int64 8B ...
    expver      (valid_time) <U4 15kB ...
Data variables:
    tp          (valid_time, latitude, longitude) float32 8MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-19T00:43 GRIB to CDM+CF via cfgrib-0.9.1...

Información de la variable tp:
{'GRIB_paramId': np.int64(228), 'GRIB_dataType': 'fc', 'GRIB_numberOfPoints': np.int64(2091), 'GRIB_typeOfLevel': 'surface',

In [11]:
# Pesos según latitud
pesos = np.cos(np.deg2rad(ds["latitude"]))

# Promedio espacial ponderado
precip = ds["tp"].weighted(pesos).mean(
    dim=["latitude", "longitude"]
)

# Convertir a DataFrame
df = precip.to_dataframe().reset_index()

df.head()

,valid_time,number,expver,tp
0,1950-01-01,0,0001,0.005010
1,1950-02-01,0,0001,0.005891
2,1950-03-01,0,0001,0.005103
3,1950-04-01,0,0001,0.007390
4,1950-05-01,0,0001,0.003994


In [12]:
# Convertir fecha
df["valid_time"] = pd.to_datetime(df["valid_time"])

# Año, mes y número de días
df["año"] = df["valid_time"].dt.year
df["mes"] = df["valid_time"].dt.month
df["dias_mes"] = df["valid_time"].dt.days_in_month

# ERA5: m/día → mm/mes
df["precipitacion_mm"] = (
    df["tp"] * 1000 * df["dias_mes"]
)

# Fecha mensual
df["fecha"] = pd.to_datetime(
    dict(
        year=df["año"],
        month=df["mes"],
        day=1
    )
)

# Dataset final
df_precipitacion = df[
    ["fecha", "año", "mes", "precipitacion_mm"]
].copy()

df_precipitacion.head()

,fecha,año,mes,precipitacion_mm
0,1950-01-01,1950,1,155.309817
1,1950-02-01,1950,2,164.955825
2,1950-03-01,1950,3,158.180623
3,1950-04-01,1950,4,221.687990
4,1950-05-01,1950,5,123.802819


In [13]:
print("Total de registros:", len(df_precipitacion))
print(
    "Periodo:",
    df_precipitacion["fecha"].min(),
    "a",
    df_precipitacion["fecha"].max()
)

print("\nValores faltantes:")
print(df_precipitacion.isnull().sum())

print(
    "\nFechas duplicadas:",
    df_precipitacion["fecha"].duplicated().sum()
)

print("\nEstadísticas:")
print(
    df_precipitacion["precipitacion_mm"].describe()
)

Total de registros: 920
Periodo: 1950-01-01 00:00:00 a 2026-08-01 00:00:00

Valores faltantes:
fecha               0
año                 0
mes                 0
precipitacion_mm    0
dtype: int64

Fechas duplicadas: 0

Estadísticas:
count    920.000000
mean     140.564987
std       78.932025
min       17.855862
25%       79.553342
50%      123.802076
75%      180.193366
max      596.482079
Name: precipitacion_mm, dtype: float64


In [14]:
fechas_esperadas = pd.date_range(
    start=df_precipitacion["fecha"].min(),
    end=df_precipitacion["fecha"].max(),
    freq="MS"
)

faltantes = fechas_esperadas.difference(
    df_precipitacion["fecha"]
)

print("Meses faltantes:", len(faltantes))

if len(faltantes) == 0:
    print("✅ Serie mensual continua")
else:
    print(faltantes)

Meses faltantes: 0
✅ Serie mensual continua


In [15]:
nombre_archivo = "precipitacion.ERA5_procesado.csv"

df_precipitacion.to_csv(
    nombre_archivo,
    index=False
)

print("✅ Archivo creado:", nombre_archivo)

✅ Archivo creado: precipitacion.ERA5_procesado.csv


In [16]:
from google.colab import files

files.download("precipitacion.ERA5_procesado.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>